# Part 1 · One melanoma biopsy → versioned input

Exploratory software, not validated patient prediction. Raw RNA stays at t0; inferred CD8 labels and assumed kinetic profiles are separate. Use a new WORK directory for each accepted preparation.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
INSTALL_PACKAGE = True
# For a local checkout: pip install -e ".[test,notebooks]" first.
# No untested hosted clone URL is executed by this notebook.
if INSTALL_PACKAGE:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[notebooks]"])
WORK = Path(os.environ.get("VTSC_NOTEBOOK_WORK", "outputs/notebook-v05"))
WORK.mkdir(parents=True, exist_ok=True)


In [ ]:
DATA_SOURCE = "melanoma"  # "synthetic" or "own"
OWN_INPUT = Path("my_biopsy.h5ad")
OWN_METADATA = None  # optional TSV with unique cell_id, cell_type, cd8_state, sample_id
OWN_LAYER = "counts"  # use "raw" for .raw.X, or None if X already contains raw counts
OWN_SAMPLE = None  # required when the source contains more than one biopsy
OWN_STATE_RECIPE = None  # explicitly choose "rank-markers-v1" if no reviewed states
SEED = 21
MAX_CELLS = 120
from vtsc.tissue_inputs import download_melanoma, synthetic_melanoma, prepare_tissue
bundle = WORK / "input.zip"
if DATA_SOURCE == "melanoma":
    record = download_melanoma(WORK / "source")
    provenance = prepare_tissue(record["input"], bundle, recipe="melanoma-markers", max_cells=MAX_CELLS, seed=SEED)
elif DATA_SOURCE == "synthetic":
    source = WORK / "synthetic_melanoma.h5ad"
    synthetic_melanoma(source, cells=MAX_CELLS, seed=SEED)
    provenance = prepare_tissue(source, bundle, max_cells=MAX_CELLS, seed=SEED, source_license="MIT; generated fixture")
elif DATA_SOURCE == "own":
    provenance = prepare_tissue(OWN_INPUT, bundle, metadata=OWN_METADATA, layer=OWN_LAYER, sample_id=OWN_SAMPLE,
                                state_recipe=OWN_STATE_RECIPE, max_cells=MAX_CELLS, seed=SEED)
else:
    raise ValueError("Choose melanoma, synthetic or own")
print(json.dumps({k:provenance[k] for k in ["sampled_composition", "sampling", "state_source", "source_license"]}, indent=2))


In [ ]:
audit = provenance["program_audit"]
print(json.dumps({"CD8_states":audit["cd8_state_counts"], "held_out_markers":audit["heldout_markers"], "coverage":audit["coverage"]}, indent=2))
(WORK / "preparation-report.json").write_text(json.dumps(provenance, indent=2))


Review Unknown and coverage. Marker concordance is a diagnostic, not validated accuracy. Missing Cytotoxic or Dysfunctional cells are not added. The 10x source is CC BY 4.0; its metadata discrepancy is recorded. SPECIAL exports remain separate and restricted; no paired outcome analysis is performed.